In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
df = pd.read_csv('../Data/Global_Terrorism_Database_May_2022.csv').sample(frac=0.1, random_state=42)


/var/folders/5l/_w0625xs2ldb8v6t7h_d5hrh0000gn/T/ipykernel_37008/1140670438.py:1: DtypeWarning: Columns (4,31,33,54,61,62,63,76,79,90,92,94,96,114,115,121) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../Data/Global_Terrorism_Database_May_2022.csv').sample(frac=0.1, random_state=42)


In [4]:
# Data Cleaning
# Drop rows with missing target values
df.dropna(subset=['gname'], inplace=True)

# Fill missing values for numerical columns with median
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

In [5]:
# Fill missing values for categorical columns with mode, excluding 'gname'
categorical_cols = df.select_dtypes(exclude=[np.number]).columns
categorical_cols = categorical_cols.drop('gname', errors='ignore')
df[categorical_cols] = df[categorical_cols].apply(lambda x: x.fillna(x.mode()[0]))

In [6]:
# Feature Engineering
# Create interaction terms using relevant features
df['interaction_term'] = df['nkill'] * df['nwound']

# One-hot encode categorical variables, excluding 'gname'
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)


/var/folders/5l/_w0625xs2ldb8v6t7h_d5hrh0000gn/T/ipykernel_37008/1086676195.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['interaction_term'] = df['nkill'] * df['nwound']


In [7]:

# Split data
X = df.drop('gname', axis=1)
y = df['gname']
train_x, test_x, train_y, test_y = train_test_split(X, y, test_size=0.25, random_state=42)

In [8]:
# Feature scaling
scaler = StandardScaler()
train_x_scaled = scaler.fit_transform(train_x)
test_x_scaled = scaler.transform(test_x)

In [9]:
# Model training with hyperparameter tuning
model = RandomForestClassifier(n_jobs=-1)
param_grid = {'n_estimators': [100, 200], 'max_depth': [10, 20]}
grid_search = GridSearchCV(model, param_grid, cv=5, scoring='roc_auc')
grid_search.fit(train_x_scaled, train_y)

/Users/ramdevpm/PythonVENV/dev/lib/python3.12/site-packages/sklearn/model_selection/_split.py:776: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(
/Users/ramdevpm/PythonVENV/dev/lib/python3.12/site-packages/sklearn/model_selection/_validation.py:982: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/Users/ramdevpm/PythonVENV/dev/lib/python3.12/site-packages/sklearn/model_selection/_validation.py", line 971, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/ramdevpm/PythonVENV/dev/lib/python3.12/site-packages/sklearn/metrics/_scorer.py", line 279, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

KeyboardInterrupt: 

: 

In [ ]:
# Evaluate model
best_model = grid_search.best_estimator_
predictions = best_model.predict(test_x_scaled)
print(classification_report(test_y, predictions))
print(f"ROC-AUC Score: {roc_auc_score(test_y, predictions)}")

# Feature importance visualization
feature_importances = best_model.feature_importances_
sns.barplot(x=feature_importances, y=X.columns)
plt.title('Feature Importance')
plt.show()